In [1]:
!docker rm -f $(docker ps -aq) || true


45e92c072cf1
4c556c724dce
3567a895d2ec
cc3ff7c6ea03
69a0178cf942
335ac9edd737
f589793d4274
358dc92b7f3b


In [4]:
!docker network prune -f
print("🧹 Entorno Docker completamente limpio.")

🧹 Entorno Docker completamente limpio.


In [6]:
%%writefile app.py
import os
import boto3
from flask import Flask, jsonify

app = Flask(__name__)

# Configuración de MinIO / S3
S3_ENDPOINT = os.getenv("S3_ENDPOINT", "http://host.docker.internal:9000")
AWS_ACCESS_KEY = os.getenv("AWS_ACCESS_KEY_ID", "minio_admin")
AWS_SECRET_KEY = os.getenv("AWS_SECRET_ACCESS_KEY", "minio_password")
BUCKET_NAME = "engine-trouble-codes"

def get_s3_client():
    return boto3.client(
        "s3",
        endpoint_url=S3_ENDPOINT,
        aws_access_key_id=AWS_ACCESS_KEY,
        aws_secret_access_key=AWS_SECRET_KEY,
        region_name="us-east-1"
    )

@app.route("/")
def home():
    return jsonify({"status": "online", "service": "engine-s3-service", "version": "next_3"})

@app.route("/seed")
def seed():
    try:
        s3 = get_s3_client()
        # Crear bucket si no existe
        buckets = [b['Name'] for b in s3.list_buckets().get('Buckets', [])]
        if BUCKET_NAME not in buckets:
            s3.create_bucket(Bucket=BUCKET_NAME)
        
        # Subir archivo de falla
        s3.put_object(
            Bucket=BUCKET_NAME,
            Key="live_fault_code.txt",
            Body="P0171 - System Too Lean (Bank 1)"
        )
        return jsonify({"message": "✅ Seeded fault code P0171 into MinIO S3!"})
    except Exception as e:
        return jsonify({"error": str(e)}), 500

@app.route("/diagnostics")
def diagnostics():
    try:
        s3 = get_s3_client()
        obj = s3.get_object(Bucket=BUCKET_NAME, Key="live_fault_code.txt")
        content = obj['Body'].read().decode('utf-8')
        return jsonify({"status": "SUCCESS", "report": content})
    except Exception as e:
        return jsonify({"error": str(e)}), 500

# IMPORTANTE: Mantener el servidor escuchando continuamente
if __name__ == "__main__":
    app.run(host="0.0.0.0", port=5000)

Overwriting app.py


In [7]:
!git add .
!git commit -m "feat: modulo next_3 servidor persistente Flask con MinIO"
!git push origin main

[main c55e521] feat: modulo next_3 servidor persistente Flask con MinIO
 3 files changed, 437 insertions(+), 134 deletions(-)
 rewrite containers/ninth_container/repaso/next_3/app.py (81%)
 create mode 100644 containers/ninth_container/repaso/next_3/numero_2.ipynb
Counting objects: 9, done.
Delta compression using up to 4 threads.
Compressing objects: 100% (9/9), done.
Writing objects: 100% (9/9), 4.43 KiB | 2.21 MiB/s, done.
Total 9 (delta 6), reused 0 (delta 0)
remote: Resolving deltas: 100% (6/6), completed with 5 local objects.
To https://github.com/leopinzon75/cloud-engineering-practice.git
   0061da4..c55e521  main -> main


In [9]:
# 1. Descargar la imagen fresca
!docker pull leopinzon75/engine-s3-service:latest

# 2. Re-eliminar cualquier contenedor previo con el mismo nombre
!docker rm -f mi_microservicio_next3 || true

# 3. Encender el servidor Flask
!docker run -d -p 5001:5000 \
  -e S3_ENDPOINT="http://host.docker.internal:9000" \
  --name mi_microservicio_next3 \
  leopinzon75/engine-s3-service:latest

latest: Pulling from leopinzon75/engine-s3-service

450697fa: Already exists 
55e0dd95: Already exists 
10ea5bd9: Already exists 
e06c2d45: Already exists 
85253995: Pulling fs layer 
6b9f97a4: Pulling fs layer 
92db1b18: Pulling fs layer 
0c030bc3: Pull complete 034kB/5.034kBBADigest: sha256:6b3bb0f771e97257ae16a2dfef5dcdf53176a13dd657ae6477d79640d2cdbd94
Status: Downloaded newer image for leopinzon75/engine-s3-service:latest
docker.io/leopinzon75/engine-s3-service:latest
mi_microservicio_next3
8a1a378e2ba9c76a1509f119609194b41af023402e078f312f330698f003a344


In [10]:
# 1. Descargar la imagen fresca
!docker pull leopinzon75/engine-s3-service:latest

# 2. Re-eliminar cualquier contenedor previo con el mismo nombre
!docker rm -f mi_microservicio_next3 || true

# 3. Encender el servidor Flask
!docker run -d -p 5001:5000 \
  -e S3_ENDPOINT="http://host.docker.internal:9000" \
  --name mi_microservicio_next3 \
  leopinzon75/engine-s3-service:latest

latest: Pulling from leopinzon75/engine-s3-service
Digest: sha256:6b3bb0f771e97257ae16a2dfef5dcdf53176a13dd657ae6477d79640d2cdbd94
Status: Image is up to date for leopinzon75/engine-s3-service:latest
docker.io/leopinzon75/engine-s3-service:latest
mi_microservicio_next3
bf54610c12c6513bf4a0bde03b287c2d4ccfa9d8e69632291092f700332a6367


In [11]:
import requests
import time

time.sleep(2)

print("--- 1. Probando / ---")
print(requests.get("http://localhost:5001/").text)

print("\n--- 2. Probando /seed ---")
print(requests.get("http://localhost:5001/seed").text)

print("\n--- 3. Probando /diagnostics ---")
print(requests.get("http://localhost:5001/diagnostics").text)

--- 1. Probando / ---


ConnectionError: HTTPConnectionPool(host='localhost', port=5001): Max retries exceeded with url: / (Caused by NewConnectionError('<urllib3.connection.HTTPConnection object at 0x10b709b10>: Failed to establish a new connection: [Errno 61] Connection refused'))

In [12]:
!docker logs mi_microservicio_next3

🛡️ Connecting to MinIO S3 at: http://host.docker.internal:9000
🚀 Misfire report uploaded to bucket 'engine-trouble-codes'!
📥 Cloud data verification: 100% Match!
💾 Permanent copy stamped to local container logs directory!


In [13]:
!docker logs mi_microservicio_next3

🛡️ Connecting to MinIO S3 at: http://host.docker.internal:9000
🚀 Misfire report uploaded to bucket 'engine-trouble-codes'!
📥 Cloud data verification: 100% Match!
💾 Permanent copy stamped to local container logs directory!


In [14]:
import subprocess
import time

# Forzar la variable S3_ENDPOINT y ejecutar tu app.py local
env = {"S3_ENDPOINT": "http://localhost:9000"}
flask_process = subprocess.Popen(
    ["python", "app.py"],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE
)

time.sleep(2)
print("🚀 Servidor Flask corriendo localmente en el puerto 5000...")

🚀 Servidor Flask corriendo localmente en el puerto 5000...


In [15]:
import requests

print("--- 1. Probando / ---")
print(requests.get("http://localhost:5000/").text)

print("\n--- 2. Probando /seed ---")
print(requests.get("http://localhost:5000/seed").text)

print("\n--- 3. Probando /diagnostics ---")
print(requests.get("http://localhost:5000/diagnostics").text)

--- 1. Probando / ---
{"service":"engine-s3-service","status":"online","version":"next_3"}


--- 2. Probando /seed ---
{"error":"Could not connect to the endpoint URL: \"http://host.docker.internal:9000/\""}


--- 3. Probando /diagnostics ---
{"error":"Could not connect to the endpoint URL: \"http://host.docker.internal:9000/engine-trouble-codes/live_fault_code.txt\""}



In [16]:
flask_process.kill()
print("🛑 Servidor previo detenido.")

🛑 Servidor previo detenido.


In [17]:
import os
import subprocess
import time

# Crear copia del entorno del sistema y sobreescribir la variable S3_ENDPOINT
custom_env = os.environ.copy()
custom_env["S3_ENDPOINT"] = "http://localhost:9000"

# Iniciar Flask con la variable de entorno correcta
flask_process = subprocess.Popen(
    ["python", "app.py"],
    env=custom_env,
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE
)

time.sleep(2)
print("🚀 Servidor Flask iniciado apuntando a http://localhost:9000")

🚀 Servidor Flask iniciado apuntando a http://localhost:9000


In [18]:
import requests

print("--- 1. Probando / ---")
print(requests.get("http://localhost:5000/").text)

print("\n--- 2. Probando /seed ---")
print(requests.get("http://localhost:5000/seed").text)

print("\n--- 3. Probando /diagnostics ---")
print(requests.get("http://localhost:5000/diagnostics").text)

--- 1. Probando / ---
{"service":"engine-s3-service","status":"online","version":"next_3"}


--- 2. Probando /seed ---
{"message":"\u2705 Seeded fault code P0171 into MinIO S3!"}


--- 3. Probando /diagnostics ---
{"report":"P0171 - System Too Lean (Bank 1)","status":"SUCCESS"}



In [19]:
!git add .
!git commit -m "feat: servidor flask persistente probado y funcionando"
!git push origin main  

[main ec9cc22] feat: servidor flask persistente probado y funcionando
 2 files changed, 446 insertions(+), 85 deletions(-)
 rewrite containers/ninth_container/repaso/next_3/app.py (78%)
Counting objects: 8, done.
Delta compression using up to 4 threads.
Compressing objects: 100% (8/8), done.
Writing objects: 100% (8/8), 10.30 KiB | 5.15 MiB/s, done.
Total 8 (delta 5), reused 0 (delta 0)
remote: Resolving deltas: 100% (5/5), completed with 5 local objects.
To https://github.com/leopinzon75/cloud-engineering-practice.git
   c55e521..ec9cc22  main -> main


In [20]:
# 1. Apagar el servidor Flask local que teníamos corriendo en Python
try:
    flask_process.kill()
    print("🛑 Servidor local detenido.")
except NameError:
    pass

# 2. Descargar la imagen actualizada desde Docker Hub
!docker pull leopinzon75/engine-s3-service:latest

# 3. Eliminar cualquier contenedor viejo
!docker rm -f mi_microservicio_next3 || true

# 4. Levantar el contenedor apuntando a MinIO en tu Mac
!docker run -d -p 5001:5000 \
  -e S3_ENDPOINT="http://host.docker.internal:9000" \
  --name mi_microservicio_next3 \
  leopinzon75/engine-s3-service:latest

🛑 Servidor local detenido.
latest: Pulling from leopinzon75/engine-s3-service

450697fa: Already exists 
55e0dd95: Already exists 
10ea5bd9: Already exists 
e06c2d45: Already exists 
5caf20fb: Pulling fs layer 
706d8476: Pulling fs layer 
ad38b329: Pulling fs layer 
c0abce23: Pull complete 034kB/5.034kBBADigest: sha256:cf329c50843c6e1aeb715fbe8c5f58a8eac8cc0394246b6d151593355e2877e1
Status: Downloaded newer image for leopinzon75/engine-s3-service:latest
docker.io/leopinzon75/engine-s3-service:latest
mi_microservicio_next3
ff72e095cde401ddc9931f09a3f520d27e5372977faa94147d94259587123c72


In [22]:
import requests
import time

time.sleep(2)

print("--- 1. Probando Contenedor Docker / ---")
print(requests.get("http://localhost:5001/").text)

print("\n--- 2. Probando /seed ---")
print(requests.get("http://localhost:5001/seed").text)

print("\n--- 3. Probando /diagnostics ---")
print(requests.get("http://localhost:5001/diagnostics").text)

--- 1. Probando Contenedor Docker / ---


ConnectionError: HTTPConnectionPool(host='localhost', port=5001): Max retries exceeded with url: / (Caused by NewConnectionError('<urllib3.connection.HTTPConnection object at 0x10c991b10>: Failed to establish a new connection: [Errno 61] Connection refused'))

In [23]:
# Ver si el contenedor está Exited o Up
!docker ps -a | grep mi_microservicio_next3

# Ver los logs del contenedor
!docker logs mi_microservicio_next3

ff72e095cde4   leopinzon75/engine-s3-service:latest   "python app.py"          2 minutes ago    Exited (0) 2 minutes ago             mi_microservicio_next3
🛡️ Connecting to MinIO S3 at: http://host.docker.internal:9000
🚀 Misfire report uploaded to bucket 'engine-trouble-codes'!
📥 Cloud data verification: 100% Match!
💾 Permanent copy stamped to local container logs directory!


In [24]:
import time

# 1. Eliminar el contenedor cerrado
!docker rm -f mi_microservicio_next3 || true

# 2. Descargar explícitamente la versión recién compilada
!docker pull leopinzon75/engine-s3-service:latest

# 3. Lanzar el contenedor
!docker run -d -p 5001:5000 \
  -e S3_ENDPOINT="http://host.docker.internal:9000" \
  --name mi_microservicio_next3 \
  leopinzon75/engine-s3-service:latest

# 4. Darle 3 segundos para iniciar la aplicación Flask
time.sleep(3)

# 5. Confirmar que sigue 'Up'
!docker ps | grep mi_microservicio_next3

mi_microservicio_next3
latest: Pulling from leopinzon75/engine-s3-service
Digest: sha256:cf329c50843c6e1aeb715fbe8c5f58a8eac8cc0394246b6d151593355e2877e1
Status: Image is up to date for leopinzon75/engine-s3-service:latest
docker.io/leopinzon75/engine-s3-service:latest
2273c6d652836790905e4ce3f6eaf7f7ef2f9325cae4e5693965a90c41fe39bf


In [25]:
import time

# 1. Eliminar el contenedor cerrado
!docker rm -f mi_microservicio_next3 || true

# 2. Descargar explícitamente la versión recién compilada
!docker pull leopinzon75/engine-s3-service:latest

# 3. Lanzar el contenedor
!docker run -d -p 5001:5000 \
  -e S3_ENDPOINT="http://host.docker.internal:9000" \
  --name mi_microservicio_next3 \
  leopinzon75/engine-s3-service:latest

# 4. Darle 3 segundos para iniciar la aplicación Flask
time.sleep(3)

# 5. Confirmar que sigue 'Up'
!docker ps | grep mi_microservicio_next3

mi_microservicio_next3
latest: Pulling from leopinzon75/engine-s3-service
Digest: sha256:cf329c50843c6e1aeb715fbe8c5f58a8eac8cc0394246b6d151593355e2877e1
Status: Image is up to date for leopinzon75/engine-s3-service:latest
docker.io/leopinzon75/engine-s3-service:latest
d533f623c4fc595d77a431113cf3121c9e704253c40595d0813290d53fcd1872


In [26]:
!docker ps | grep mi_microservicio_next3

In [27]:
!docker ps --filter "name=mi_microservicio_next3"

CONTAINER ID   IMAGE     COMMAND   CREATED   STATUS    PORTS     NAMES


In [28]:
!docker ps -a | grep -i "mi_microservicio_next"

d533f623c4fc   leopinzon75/engine-s3-service:latest   "python app.py"          3 minutes ago    Exited (0) 3 minutes ago             mi_microservicio_next3


In [29]:
import requests

print("--- 1. Probando / ---")
print(requests.get("http://localhost:5001/").text)

print("\n--- 2. Probando /seed ---")
print(requests.get("http://localhost:5001/seed").text)

print("\n--- 3. Probando /diagnostics ---")
print(requests.get("http://localhost:5001/diagnostics").text)

--- 1. Probando / ---


ConnectionError: HTTPConnectionPool(host='localhost', port=5001): Max retries exceeded with url: / (Caused by NewConnectionError('<urllib3.connection.HTTPConnection object at 0x10c789b10>: Failed to establish a new connection: [Errno 61] Connection refused'))

In [30]:
# 1. Ver el estado de todos los contenedores (activos y detenidos)
!docker ps -a --filter "name=mi_microservicio_next3"

print("\n" + "="*50 + "\n")

# 2. Leer las últimas líneas del registro del contenedor
!docker logs --tail 20 mi_microservicio_next3

CONTAINER ID   IMAGE                                  COMMAND           CREATED         STATUS                     PORTS     NAMES
d533f623c4fc   leopinzon75/engine-s3-service:latest   "python app.py"   6 minutes ago   Exited (0) 5 minutes ago             mi_microservicio_next3


🛡️ Connecting to MinIO S3 at: http://host.docker.internal:9000
🚀 Misfire report uploaded to bucket 'engine-trouble-codes'!
📥 Cloud data verification: 100% Match!
💾 Permanent copy stamped to local container logs directory!


In [31]:
!docker rm -f mi_microservicio_next3 || true
!docker rmi leopinzon75/engine-s3-service:latest || true
print("🧹 Imagen y contenedor antiguos eliminados de la memoria local.")

mi_microservicio_next3
Untagged: leopinzon75/engine-s3-service:latest
Untagged: leopinzon75/engine-s3-service@sha256:cf329c50843c6e1aeb715fbe8c5f58a8eac8cc0394246b6d151593355e2877e1
Deleted: sha256:f01950ca526da2bc2801fe38930082c45aa4ba777edb13bd96ae7df46986559f
Deleted: sha256:9855016e1e17a504b56338455cf72d5f35851018b3e2e108dae1b4a6c794927f
Deleted: sha256:8aafad2d83548141c5964dc2051e9dc7dc74bddf4b335ba9822e41e1c3b15aee
Deleted: sha256:510a9acab1cb5265463d130368173867342ab97765a4b1e04afe8e02e34bb266
Deleted: sha256:3980bf00f9b8a9483e4c15ad9e42c22720e437db84fc5024fc3ce0126815427f
🧹 Imagen y contenedor antiguos eliminados de la memoria local.


In [33]:
# Construir la imagen localmente usando el código de la carpeta next_3
!docker build -t mi_microservicio_local:v1 ./containers/ninth_container/repaso/next_3

unable to prepare context: path "./containers/ninth_container/repaso/next_3" not found


In [34]:
!docker build -t mi_microservicio_local:v1 .

[+] Building 0.0s (0/1)                                                         
[+] Building 0.2s (1/1)                                                         
 => [internal] load build definition from Dockerfile                       0.1s
 => => transferring dockerfile: 210B                                       0.0s
[+] Building 0.3s (2/2)                                                         
 => [internal] load build definition from Dockerfile                       0.1s
 => => transferring dockerfile: 210B                                       0.0s
 => [internal] load .dockerignore                                          0.0s
 => => transferring context: 2B                                            0.0s
[+] Building 0.5s (2/3)                                                         
 => [internal] load build definition from Dockerfile                       0.1s
 => => transferring dockerfile: 210B                                       0.0s
 => [internal] load .dockerignore   

In [35]:
import time

# Lanzar el contenedor creado localmente
!docker run -d -p 5001:5000 \
  -e S3_ENDPOINT="http://host.docker.internal:9000" \
  --name mi_microservicio_next3 \
  mi_microservicio_local:v1

time.sleep(3)

# Verificar que el estado esté en 'Up' y NO en 'Exited'
!docker ps --filter "name=mi_microservicio_next3"

c5453857b1fd7e30b8177e977a1e37fdb5750057a3492d2e693843e56a6d23fb
CONTAINER ID   IMAGE                       COMMAND           CREATED         STATUS         PORTS                    NAMES
c5453857b1fd   mi_microservicio_local:v1   "python app.py"   5 seconds ago   Up 3 seconds   0.0.0.0:5001->5000/tcp   mi_microservicio_next3


In [36]:
import requests

print("--- 1. Probando / ---")
print(requests.get("http://localhost:5001/").text)

print("\n--- 2. Probando /seed ---")
print(requests.get("http://localhost:5001/seed").text)

print("\n--- 3. Probando /diagnostics ---")
print(requests.get("http://localhost:5001/diagnostics").text)

--- 1. Probando / ---
🌐 Cloud-Connected Diagnostic Portal Active

--- 2. Probando /seed ---
✅ Seeded fault code P0171 into MinIO S3!

--- 3. Probando /diagnostics ---
📊 Data Retrieved From Cloud Bucket:🚨 CLOUD DATA: P0171 - System Too Lean (Bank 1)


In [ ]:
--- 1. Probando / ---
🌐 Cloud-Connected Diagnostic Portal Active

--- 2. Probando /seed ---
✅ Seeded fault code P0171 into MinIO S3!

--- 3. Probando /diagnostics ---
📊 Data Retrieved From Cloud Bucket:🚨 CLOUD DATA: P0171 - System Too Lean (Bank 1)